# Inference FLOPs: H-balanced model vs AME

Companion to [`../concepts/h-balanced-model.md`](../concepts/h-balanced-model.md) and
[`../concepts/ame-model.md`](../concepts/ame-model.md). AME (Attention Memory Embedding, formerly MAE)
is the attention method in [`src/core/ame.py`](../../src/core/ame.py); its live record is
[`../notes/ame_implementation.md`](../notes/ame_implementation.md), and its design record is
`~/code/research/ame/ame.md` (outside this repo), whose §2 and §4.4 price its decode.

An H-balanced transformer is fixed by one even integer `H`:

| | rule |
|---|---|
| layers | `n_blocks = H` |
| heads | `q_heads = H`, `kv_heads = H/2` (GQA, always) |
| residual width | `dmodel = 64 H`, so `head_dim = 64` |
| FF hidden | `dff = 4 dmodel = 256 H`, gated (`gate`, `up`, `down`) |
| vocab | GPT-NeoX tokenizer: 50,277 ids, `vocab_size = 50,304` in the configs (padded to a multiple of 128) |

**AME** at the same `H` is that model with three changes to every attention layer, as built in
`src/core/ame.py`:

* **No `q_proj` or `k_proj`.** Queries and keys are rows of two banks, `E_Q` and `E_K`, one row of
  `datt` per token id, gathered by id. The banks are parameters - `2 V datt` per layer, `128 V H²` in
  all, 4.9x the rest of the model at `H = 16` and 1.1x at `H = 100` - but a gather is not a matmul,
  so like the embedding table they cost no FLOPs. Nor do AME's sink vectors and per-head scalars
  (`2 datt + 5 heads` per layer).
* **A full-width `v_proj`.** AME is MHA-shaped: every query head has its own value head, so `v_proj`
  is `dmodel x datt`, twice GQA's. Net, the attention projections drop from `3 d²` to `2 d²` per
  layer.
* **Scores from token ids and distance only**, `s_ij = tau · Ê_Q[t_i] · Ê_K[t_j] - m (i - j)`, which is
  what makes its decode cheap.

What that does to the score path depends on the pass:

* **A prefill is ordinary causal attention** over the gathered rows - the same `QKᵀ` and `A@V` pair
  work as the vanilla model (design §2.2: "identical cost and kernels to standard attention").
  `AME_PREFILL_COST` scales it; 1 is AME as designed.
* **A decode step has no `QKᵀ`**: the score of key `j` is an entry of the query token's **score
  row** `R = T[t_i, ·]`, its row of `T = Ê_Q Ê_Kᵀ`, the table of every query type against every key
  type - one `V x V` table per head and layer. These counts take `T` as **precomputed**, so reading
  a row costs no FLOPs (`AME_ROW_MISS = 0`). What a step still pays is the **value sum over the
  top-k positions** it selects (`AME_TOP_K = 2048`, design §4.4) and, because the KV shift makes a
  key's score `α₁ R[t_j] + α₂ R[t_{j-1}]`, **scoring every cached bigram for the top-k** (`AME_SHIFT`,
  on by default as in the settled design and the best Stage 2 variant; `ame_implementation.md`, "At
  decode").
* **Without a stored table** a step computes its row from the banks: one `d_head` dot product per
  head for every distinct token type in the context. That is the design record's own plan (§2.1): the
  full table is `V²` entries per head and layer, 5.06 GB in bf16 and 50.6 TB at `H = 100`, so it keeps
  only the banks and caches the rows of frequent query tokens. `AME_ROW_MISS` is the share of steps
  that compute their row; the sweeps further down draw 0 to 100%.

**AMEm** (`variant="amem"`) is AME with what those changes save - `d²` of attention weights per
layer, net of the wider `v_proj` - spent on a wider FF: `dff = 4 dmodel + dmodel / 3`, rounded to a
multiple of 32. That matches the vanilla parameter count outside the banks to within 0.15% at the
sizes drawn here, and exactly at `H = 24`, so AMEm does the vanilla model's weight work per token and
isolates what AME's attention costs. It appears in its own two figures and in the methods figures.

This notebook's first version charged AME no attention cost in either pass. That is still one
setting away - all four AME knobs off - but the design record and the implementation have since
established the costs above, and they are the defaults.

## The counting model

**Weights: 2 FLOPs per active parameter per token** (one multiply + one add per MAC).
Active = every parameter that takes part in a matmul for that token:

* attention projections (`q, k, v, o` for the vanilla model, `v, o` for AME) and the three gated-FF
  matrices (`gate`, `up`, `down`), in every layer;
* the LM head (`vocab x dmodel`);
* **not** the embedding table, AME's Q/K banks or its per-head extras - a lookup is a gather, not a
  matmul, so they contribute parameters but no FLOPs.

**Attention is counted exactly, not as "2 x params".** The score path has no weights; its cost is
quadratic in the context and is what makes long contexts expensive. With causal masking only the
`i >= j` pairs are computed:

```
pairs(L)  = L(L+1)/2                       # prefill over a context of L tokens
pairs     = L                              # one decode step attending to L cached keys

QK^T  : 2 * head_dim FLOPs per (query, key, head)   -> 2 * head_dim * q_heads * pairs
A @ V : 2 * head_dim FLOPs per (query, key, head)   -> 2 * head_dim * q_heads * pairs
            per layer: 4 * datt * pairs            (datt = head_dim * q_heads)

AME prefill: the same, times AME_PREFILL_COST
AME decode : 2 * datt * min(L, top_k)              A @ V over the selected positions, no QK^T
           + 2 * datt * U(L) * AME_ROW_MISS        rows computed from the banks; 0 with the table
           + 3 * q_heads * L     if AME_SHIFT      every cached bigram scored for the top-k
```

The row term is zero by default, since the table is precomputed. When rows are computed instead,
`U(L)` is the number of distinct token types among the `L` cached tokens: `2.55 L^0.76`, capped at
`L` and the vocab - a Heaps'-law fit to the design's Zipf estimates (§4.2: ~0.6-1k types at 2k tokens,
5-10k at 32k, 10-20k at 100k), extrapolated past 100k. It reaches the whole vocab at about 450k
tokens, which bends those curves there. A bigram's score is two reads of the row, weighted and
summed - 3 FLOPs per head with `tau` folded into the `α`s - and the bigram count is taken as the
position count, as the implementation note does.

GQA does **not** reduce the score path: the `H/2` kv heads are broadcast to all `H` query heads, so
it runs `q_heads` times. GQA only shrinks the `k`/`v` projections (and the KV cache).

**Excluded** (together below ~1% at any `H` we use, and all linear in tokens): RMSNorm/LayerNorm and
AME's QK-norm, residual additions, the RoPE rotation, AME's distance bias, the KV-shift mixing of the
cached values and the sink column, the activation function, the top-k merge itself (comparisons, not
FLOPs), and the softmax - the last of these is available behind `softmax_flops_per_score` if you want
it counted.

In [ ]:
from dataclasses import dataclass, asdict
from math import log10

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

HEAD_DIM = 64  # fixed by the H-balanced rule: dmodel = 64H and q_heads = H
VARIANTS = ("vanilla", "ame", "amem")

## Configuration

Everything you would want to change lives in the next cell. Edit it, run the notebook top to
bottom, and every figure, table and check below follows - the four AME knobs in particular,
`AME_PREFILL_COST`, `AME_TOP_K`, `AME_ROW_MISS` and `AME_SHIFT`. The prose quotes the defaults.

In [ ]:
# ----- the one cell to edit -------------------------------------------------------------------
# AME's attention cost, as the design record and the implementation note price it (ame.md 2 and
# 4.4, ame_implementation.md "At decode"), with its vocab x vocab score table precomputed. All four
# at 0 / False charge no attention cost at all, which is what this notebook assumed before the AME
# design was written down.
AME_PREFILL_COST = 1.0       # share of the dense score path an AME prefill pays: 1 = as designed
AME_TOP_K = 2048             # positions whose values one AME decode step sums, per head; None = all
AME_ROW_MISS = 0.0           # share of decode steps that compute the query token's score row from
                             # the banks instead of reading it from the precomputed V x V table:
                             # 0 = the whole table is precomputed (the assumption here, 2026-11-13),
                             # 1 = no table, every row computed (ame.md 2.1 stores only the banks)
AME_SHIFT = True             # the KV shift (settled, and in the best Stage 2 variant): the top-k is
                             # then over bigrams, scored at 3 FLOPs per cached position per head
HEAPS = (2.55, 0.76)         # distinct token types among L tokens = 2.55 * L^0.76, capped at L and V
                             # (a fit to ame.md 4.2's Zipf estimates, ~0.8k, 7.5k, 15k at 2k, 32k,
                             # 100k): the row entries a step computes, when AME_ROW_MISS > 0
H_VALUES = [16, 20, 24, 32, 40, 50]          # model sizes drawn in every figure
CONTEXT_RANGE = (256, 1 << 20)               # span of the context axis: 256 .. 1,048,576 tokens
CONTEXT_POINTS = 100         # log-spaced samples across it, so hovering reads values in between
VOCAB = 50_304               # GPT-NeoX: 50,277 ids padded to a multiple of 128 (configs/_model/h_balanced)
GATED_FF = True              # gate + up + down; False counts the ungated two-matrix FF
H_PICK = 16                  # the size broken down in the "where the FLOPs go" figures
CONTEXTS_BD = [1024, 8192, 65536, 1048576]   # the contexts those figures show
SWEEP_ROW_MISS = [0.0, 0.14, 0.25, 0.5, 1.0]  # shares of rows computed, in the sweep figures: 0.14
                                              # and 0.25 are ame.md 4.4's 10k- and 3k-type LRU row
                                              # caches, its alternative to storing the table
SWEEP_PAIRS = [(50, 50), (50, 40), (100, 80), (252, 200)]  # (AME H, vanilla H) drawn in the sweeps
METHOD_PAIR = (100, 80)      # (AME H, vanilla H) for the long-context attention-method figures
THEME = "light"              # "light" or "dark"
# ------------------------------------------------------------------------------------------------

# every inference_flops call below passes these; the vanilla model ignores the AME ones
KW = dict(prefill_cost=AME_PREFILL_COST, top_k=AME_TOP_K, row_miss=AME_ROW_MISS, shift=AME_SHIFT,
          heaps=HEAPS, gated_ff=GATED_FF)

_ratio = CONTEXT_RANGE[1] / CONTEXT_RANGE[0]
CONTEXTS = sorted({int(round(CONTEXT_RANGE[0] * _ratio ** (i / (CONTEXT_POINTS - 1))))
                   for i in range(CONTEXT_POINTS)})          # the sampled x values
CONTEXT_TICKS = [2 ** k for k in range(4, 31)                 # ... labelled at the powers of two
                 if CONTEXT_RANGE[0] <= 2 ** k <= CONTEXT_RANGE[1]]

VANILLA = "vanilla"
AME = "AME"
AMEM = "AMEm"
FAMILIES = [(VANILLA, "vanilla"), (AME, "ame")]        # the two families in most figures
FAMILIES3 = FAMILIES + [(AMEM, "amem")]                # AMEm joins in its own two figures
FF_NOTE = "gated FF" if GATED_FF else "ungated FF"
TOPK_NOTE = ("every value" if AME_TOP_K is None else "no values" if AME_TOP_K == 0
             else f"top-{AME_TOP_K:,} values")
ROW_NOTE = ("score rows read from the precomputed table" if AME_ROW_MISS == 0
            else f"score row computed from the banks on {AME_ROW_MISS:.0%} of steps")
AME_NOTE = f"AME decode: {TOPK_NOTE}, {ROW_NOTE}" + (", bigram selection" if AME_SHIFT else "")

In [ ]:
def h_balanced_dims(H: int) -> dict:
    """Every dimension of an H-balanced model, derived from H alone. GQA throughout."""
    if H % 2 != 0:
        raise ValueError(f"H must be divisible by 2, got {H}")
    dmodel = 64 * H
    return dict(H=H, n_blocks=H, dmodel=dmodel, datt=dmodel, dff=4 * dmodel,
                q_heads=H, kv_heads=H // 2, head_dim=HEAD_DIM)


def weight_params(vocab_size: int, H: int, *, variant: str = "vanilla",
                  gated_ff: bool = True) -> dict:
    """Parameters by role. Everything except the embedding table, AME's Q/K banks and AME's
    per-head extras takes part in a matmul; those are gathers or scalars, so they cost no FLOPs.

    variant="ame"  drops q_proj and k_proj and adds two banks per layer, E_Q and E_K, each one row
                   of datt per token id. AME is MHA-shaped - every query head has its own value
                   head - so its v_proj is the full dmodel x datt, not GQA's half width
                   (src/core/ame.py ignores kv_heads). Its extras are a key and a value sink of
                   datt each and alpha, beta and tau, 5 scalars per head.
    variant="amem" is AME with dff widened by the weights that change saves, d^2 per layer, so
                   that everything except the banks matches the vanilla model of the same H:
                   +d/3 with a gated FF, +d/2 ungated, rounded to a multiple of 32.
    """
    if variant not in VARIANTS:
        raise ValueError(f"variant must be one of {VARIANTS}, got {variant!r}")
    d = h_balanced_dims(H)
    dmodel, datt, dff, L = d["dmodel"], d["datt"], d["dff"], d["n_blocks"]
    kv_width = d["head_dim"] * d["kv_heads"]
    if variant == "vanilla":
        attn_per_layer = dmodel * datt + 2 * dmodel * kv_width + datt * dmodel   # q, k, v, o
        banks = extras = 0
    else:
        attn_per_layer = dmodel * datt + datt * dmodel                           # v, o
        banks = L * 2 * vocab_size * datt                                        # E_Q, E_K
        extras = L * (2 * datt + 5 * d["q_heads"])                               # sinks, scalars
    ff_matrices = 3 if gated_ff else 2                     # gate + up + down
    if variant == "amem":           # spend the deleted q_proj/k_proj, net of the wider v_proj
        freed = 2 * dmodel * kv_width                       # = d^2 per layer
        dff += 32 * round(freed / (ff_matrices * dmodel) / 32)
    return dict(
        attn_projections=L * attn_per_layer,
        feedforward=L * ff_matrices * dmodel * dff,
        lm_head=vocab_size * dmodel,
        embedding_table=vocab_size * dmodel,  # parameters, but no FLOPs (gather)
        qk_banks=banks,                       # likewise
        ame_extras=extras,                    # and no matmul either
    )


def total_params(vocab_size: int, H: int, *, variant: str = "vanilla", gated_ff: bool = True,
                 banks: bool = True) -> int:
    """All parameters, counted as nano trains them: untied embedding and head, norms, no biases.

    banks=False leaves out AME's Q/K banks, i.e. counts everything else ("the backbone").
    """
    p = weight_params(vocab_size, H, variant=variant, gated_ff=gated_ff)
    if not banks:
        p.pop("qk_banks")
    d = h_balanced_dims(H)
    norms = d["n_blocks"] * 2 * d["dmodel"] + d["dmodel"]
    return sum(p.values()) + norms

In [ ]:
@dataclass(frozen=True)
class Flops:
    """FLOPs of one forward pass, split by term. All counts are for the whole pass."""

    H: int
    vocab_size: int
    context_length: int
    variant: str           # "vanilla", "ame" or "amem"
    mode: str              # "prefill" (L tokens at once) or "decode" (1 token, cache of L)
    tokens: int            # tokens produced by this pass
    attn_projections: int  # q/k/v/o matmuls (v/o for AME)
    attn_scores: int       # pair work: 2*head_dim per (query, key, head) for QK^T and again for
                           # A@V. An AME decode step does A@V over its top-k only, and no QK^T
    score_rows: int        # AME decode only: computing the query token's score row from the banks
    selection: int         # AME decode with the KV shift: every cached bigram scored for the top-k
    softmax: int
    feedforward: int
    lm_head: int

    @property
    def total(self) -> int:
        return (self.attn_projections + self.attn_scores + self.score_rows + self.selection
                + self.softmax + self.feedforward + self.lm_head)

    @property
    def per_token(self) -> float:
        return self.total / self.tokens

    @property
    def attention_share(self) -> float:
        """Fraction of the pass spent on attention scores: the pair work plus AME's score rows
        and selection."""
        return (self.attn_scores + self.score_rows + self.selection) / self.total

    def as_row(self) -> dict:
        row = dict(asdict(self))
        row.update(total=self.total, per_token=self.per_token,
                   attention_share=self.attention_share)
        return row


def distinct_types(context_length: int, vocab_size: int, heaps=(2.55, 0.76)) -> int:
    """Distinct token types among `context_length` tokens: a Heaps'-law estimate a * L^b, capped
    at the context length and the vocab."""
    a, b = heaps
    return min(context_length, vocab_size, round(a * context_length ** b))


def inference_flops(
    vocab_size: int,
    H: int,
    context_length: int,
    *,
    mode: str = "prefill",
    variant: str = "vanilla",
    prefill_cost: float = 1.0,
    top_k: int | None = 2048,
    row_miss: float = 0.0,
    shift: bool = True,
    heaps: tuple = (2.55, 0.76),
    gated_ff: bool = True,
    softmax_flops_per_score: int = 0,
) -> Flops:
    """FLOPs of one inference forward pass of an H-balanced model, or of AME / AMEm at the same H.

    Weights cost 2 FLOPs per active parameter per token; the attention score path is counted
    exactly over the causal (i >= j) pairs. See the markdown cell above for what is excluded.

    mode="prefill": the whole context of `context_length` tokens in one pass.
    mode="decode" : one new token attending to `context_length` cached positions (incl. itself).

    The AME arguments (ignored for the vanilla model):
      prefill_cost  share of the dense score path an AME prefill pays. 1.0 is AME as designed:
                    a prefill is ordinary causal attention over the gathered bank rows.
      top_k         positions whose values one AME decode step sums, per head (None = all). A
                    decode step has no QK^T: a key's score is an entry of the query token's row
                    of the V x V score table.
      row_miss      share of decode steps that compute that row from the banks - one d_head dot
                    product per head for every distinct type in the context - instead of
                    reading it from the precomputed table (0.0: the whole table is stored).
      shift         the KV shift. A key's score is then a1*R[t_j] + a2*R[t_j-1], so the top-k
                    index is keyed by bigram and a decode step scores every cached position:
                    two row reads, weighted and summed, 3 FLOPs per position per head. Without
                    it each type's occurrences are already in score order and the selection
                    is comparisons only.
      heaps         (a, b) of the distinct-type estimate, see `distinct_types`.
    """
    if variant not in VARIANTS:
        raise ValueError(f"variant must be one of {VARIANTS}, got {variant!r}")
    if not (0.0 <= prefill_cost <= 1.0 and 0.0 <= row_miss <= 1.0):
        raise ValueError("prefill_cost and row_miss must be in [0, 1]")
    if top_k is not None and top_k < 0:
        raise ValueError(f"top_k must be None or >= 0, got {top_k}")
    if context_length < 1:
        raise ValueError("context_length must be >= 1")
    d = h_balanced_dims(H)
    p = weight_params(vocab_size, H, variant=variant, gated_ff=gated_ff)
    layers, heads, datt = d["n_blocks"], d["q_heads"], d["datt"]

    if mode == "prefill":
        tokens = context_length
        pairs = context_length * (context_length + 1) // 2
    elif mode == "decode":
        tokens = 1
        pairs = context_length
    else:
        raise ValueError(f"mode must be 'prefill' or 'decode', got {mode!r}")

    rows = selection = 0
    if variant == "vanilla":
        scored = pairs
        scores = layers * 4 * datt * pairs                         # QK^T + A@V, every pair
    elif mode == "prefill":
        scored = prefill_cost * pairs
        scores = round(prefill_cost * layers * 4 * datt * pairs)   # the same, scaled
    else:
        scored = pairs if top_k is None else min(top_k, pairs)
        scores = layers * 2 * datt * scored                        # A@V over the kept positions
        rows = round(row_miss * layers * 2 * datt
                     * distinct_types(context_length, vocab_size, heaps))
        selection = layers * 3 * heads * context_length if shift else 0

    return Flops(
        H=H,
        vocab_size=vocab_size,
        context_length=context_length,
        variant=variant,
        mode=mode,
        tokens=tokens,
        attn_projections=2 * p["attn_projections"] * tokens,
        attn_scores=scores,
        score_rows=rows,
        selection=selection,
        softmax=round(layers * heads * scored * softmax_flops_per_score),
        feedforward=2 * p["feedforward"] * tokens,
        lm_head=2 * p["lm_head"] * tokens,
    )

## Sanity checks

Properties the formulas must have, asserted rather than eyeballed.

In [ ]:
# 1. Weight terms are exactly 2 FLOPs per active parameter per token, in every family.
for v in VARIANTS:
    f, p = inference_flops(50_304, 16, 1024, variant=v), weight_params(50_304, 16, variant=v)
    assert f.attn_projections == 2 * p["attn_projections"] * 1024
    assert f.feedforward == 2 * p["feedforward"] * 1024
    assert f.lm_head == 2 * p["lm_head"] * 1024

# 2. Attention scores are quadratic, weight terms linear: doubling L ~4x vs exactly 2x.
a, b = inference_flops(50_304, 16, 4096), inference_flops(50_304, 16, 8192)
assert b.feedforward == 2 * a.feedforward
assert 3.9 < b.attn_scores / a.attn_scores < 4.0  # L(L+1) ratio, just under 4

# 3. A vanilla prefill of L tokens costs the same as L decode steps over the same growing cache.
L = 128
step_sum = sum(inference_flops(50_304, 8, i, mode="decode").total for i in range(1, L + 1))
assert step_sum == inference_flops(50_304, 8, L).total

# 4. AME has no q_proj or k_proj and a full-width v_proj: 2d^2 of attention projections per layer
#    against vanilla's 3d^2, FF and head untouched. Closed forms for the non-embedding parameters:
#    61440*H^3 for vanilla, 57344*H^3 + 133*H^2 (sinks and scalars) for the AME backbone, and
#    128*V*H^2 of banks.
van = inference_flops(50_304, 24, 4096)
ame = inference_flops(50_304, 24, 4096, variant="ame")
assert 3 * ame.attn_projections == 2 * van.attn_projections
assert ame.feedforward == van.feedforward and ame.lm_head == van.lm_head
for H in (12, 16, 50):
    emb = 2 * 50_304 * 64 * H
    assert weight_params(50_304, H, variant="ame")["qk_banks"] == 128 * 50_304 * H ** 2
    backbone = total_params(50_304, H, variant="ame", banks=False) - emb
    assert abs(backbone - 57344 * H ** 3 - 133 * H ** 2) < 0.001 * 57344 * H ** 3
    assert abs(total_params(50_304, H) - emb - 61440 * H ** 3) < 0.001 * 61440 * H ** 3

# 4b. ... which is exactly what nano trains: the Stage 2 winner S2-PS (AME at H = 12, ALiBi and the
#     shift) has 1,103,599,056 parameters (stage_2_report.md), and the Stage 1 pair in
#     ame_experiments.md has a 106.2M dense backbone, a 99.1M AME one and 927.2M of banks.
assert total_params(50_304, 12, variant="ame") == 1_103_599_056
emb12 = 2 * 50_304 * 768
assert round((total_params(50_304, 12) - emb12) / 1e5) == 1062
assert round((total_params(50_304, 12, variant="ame", banks=False) - emb12) / 1e5) == 991
assert round(weight_params(50_304, 12, variant="ame")["qk_banks"] / 1e5) == 9272

# 4c. AMEm spends the d^2 per layer on dff, so everything but the banks and AME's extras matches
#     vanilla: exactly where d/3 is a multiple of 32 (H = 24) or the FF is ungated, and to 0.2%
#     elsewhere - so it does the vanilla model's weight work per token.
def matched(H, gated_ff=True):
    p = weight_params(50_304, H, variant="amem", gated_ff=gated_ff)
    return total_params(50_304, H, variant="amem", gated_ff=gated_ff, banks=False) - p["ame_extras"]

assert matched(24) == total_params(50_304, 24)
for H in (12, 16, 20, 50, 100):
    assert matched(H, gated_ff=False) == total_params(50_304, H, gated_ff=False)
    assert abs(matched(H) / total_params(50_304, H) - 1) < 0.002
amem = inference_flops(50_304, 24, 4096, variant="amem")
assert amem.attn_projections + amem.feedforward == van.attn_projections + van.feedforward

# 5. An AME prefill is ordinary attention, so it does the vanilla pair work; prefill_cost scales it.
assert ame.attn_scores == van.attn_scores and ame.score_rows == ame.selection == 0
assert inference_flops(50_304, 24, 4096, variant="ame", prefill_cost=0.25).attn_scores == \
    van.attn_scores // 4

# 6. An AME decode step has no QK^T, so summing every value is half the vanilla pair work. With
#    the score table precomputed (the default) and the top-k full, the only term still growing
#    with L is the bigram selection: 3 FLOPs per position per head per layer, none without the
#    shift. Computing the rows from the banks instead adds a term that stops growing once the
#    context holds every type.
d_van = inference_flops(50_304, 16, 1000, mode="decode")
d_all = inference_flops(50_304, 16, 1000, mode="decode", variant="ame", top_k=None)
assert 2 * d_all.attn_scores == d_van.attn_scores and d_all.score_rows == 0
far = [inference_flops(50_304, 16, L, mode="decode", variant="ame") for L in (2048, 1 << 20)]
assert far[1].total - far[0].total == 3 * 16 * 16 * ((1 << 20) - 2048)
flat = [inference_flops(50_304, 16, L, mode="decode", variant="ame", shift=False)
        for L in (2048, 1 << 20)]
assert flat[0].total == flat[1].total
computed = [inference_flops(50_304, 16, L, mode="decode", variant="ame", row_miss=1)
            for L in (1 << 20, 1 << 21)]
assert computed[1].total - computed[0].total == 3 * 16 * 16 * (1 << 20)
assert computed[0].score_rows == computed[1].score_rows > 0

# 7. All four AME knobs off charge no attention cost at all, so the cost is exactly linear in
#    tokens - the assumption this notebook started from, before the design priced the decode.
FREE = dict(variant="ame", prefill_cost=0.0, top_k=0, row_miss=0.0, shift=False)
free = [inference_flops(50_304, 16, L, **FREE) for L in (8192, 16_384)]
assert all(f.attn_scores == f.score_rows == f.selection == 0 for f in free)
assert free[1].total == 2 * free[0].total
assert inference_flops(50_304, 16, 8192, mode="decode", **FREE).attention_share == 0

print("all checks pass")
for v in VARIANTS:
    f = inference_flops(VOCAB, 16, 2048, variant=v, **KW)
    banks = weight_params(VOCAB, 16, variant=v)["qk_banks"]
    print(f"H=16 {v:>7}: {total_params(VOCAB, 16, variant=v, gated_ff=GATED_FF, banks=False) / 1e6:7.1f}M"
          f" params{f' + {banks / 1e6:,.0f}M of banks' if banks else ''} | 2048-token prefill "
          f"{f.total / 1e12:.2f} TFLOPs ({f.per_token / 1e9:.2f} GFLOPs/token), "
          f"attention score path {f.attention_share:.1%}")

## Figures

Both model families are drawn together over the same `H` values: **colour is `H`, line style is the
family** (solid = vanilla, dashed = AME). Colour is keyed to the vanilla parameter count; AME of the
same `H` has a backbone 4.7-6.4% smaller at these sizes (`57344 H³` against `61440 H³` outside the
embedding and head) plus its banks, and its full label - `0.34B + 1.6B of banks` at `H = 16` - is used
wherever AME is drawn on its own. Every line figure carries a **log / linear switch for its value
axis** - the buttons above the plot relayout it in place, direct labels included (the stacked bar
charts have no such switch: their value axis is a share and their other axis is categorical). The
context axis carries `CONTEXT_POINTS` log-spaced samples (100 by default) and is labelled at the
powers of two, so hovering anywhere between the labels reads out real values. One chart surface,
light or dark: set `THEME` and re-run. The series palette is the validated categorical order,
assigned to model sizes in fixed order and never cycled; the vanilla line of each size is also
labelled directly at its right end, so identity never rests on colour alone. The table under the
first figure is the same data in readable form.

In [ ]:
PALETTE = {
    "light": dict(
        surface="#fcfcfb", text="#0b0b0b", muted="#52514e", grid="#e6e5e1",
        series=["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"],
        series_long=["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300",
                     "#9a5ce0", "#00a0b0", "#b02a37", "#7a5c2e"],
    ),
    "dark": dict(
        surface="#1a1a19", text="#f0efec", muted="#c3c2b7", grid="#34332f",
        series=["#3987e5", "#d95926", "#199e70", "#c98500", "#d55181", "#008300"],
        series_long=["#3987e5", "#d95926", "#199e70", "#c98500", "#d55181", "#008300",
                     "#a970ea", "#19aab8", "#e06c75", "#b08a52"],
    ),
}
T = PALETTE[THEME]


def size_label(H, variant="vanilla"):
    """A model's parameter count in billions, two decimals: the series name. AME's Q/K banks are
    named separately - they are parameters, but gathers, so they cost no FLOPs."""
    backbone = total_params(VOCAB, H, variant=variant, gated_ff=GATED_FF, banks=False) / 1e9
    if variant == "vanilla":
        return f"{backbone:.2f}B"
    banks = weight_params(VOCAB, H, variant=variant, gated_ff=GATED_FF)["qk_banks"] / 1e9
    return f"{backbone:.2f}B + {banks:.1f}B of banks"


SIZES = {H: size_label(H) for H in H_VALUES}   # vanilla parameter count per H, used as the key


def style(fig, *, title, subtitle=None, xtitle=None, ytitle=None, legend_title=None):
    """Recessive axes, readable ink, one consistent surface."""
    head = title if subtitle is None else f"{title}<br><span style='font-size:13px;color:{T['muted']}'>{subtitle}</span>"
    fig.update_layout(
        title=dict(text=head, font=dict(size=18, color=T["text"]), x=0, xanchor="left", y=0.96,
                   yanchor="top"),
        paper_bgcolor=T["surface"], plot_bgcolor=T["surface"],
        font=dict(family="Inter, system-ui, sans-serif", size=13, color=T["muted"]),
        legend=dict(title=legend_title or "", bgcolor="rgba(0,0,0,0)",
                    font=dict(color=T["text"]), yanchor="top", y=1, xanchor="left", x=1.02),
        margin=dict(l=70, r=140, t=130, b=60), hovermode="x unified", height=520,
    )
    fig.update_xaxes(title=xtitle, showgrid=False, zeroline=False,
                     linecolor=T["grid"], ticks="outside", tickcolor=T["grid"])
    fig.update_yaxes(title=ytitle, gridcolor=T["grid"], zeroline=False, linecolor="rgba(0,0,0,0)")
    return fig


def family_legend(fig, families):
    """Colour carries the model size, dash the family: one entry per size, plus one key per family.

    `families` is [(label, dash), ...]; the first one carries the per-size legend entries.
    """
    primary = families[0][0]
    for tr in fig.data:
        h_label, family = (part.strip() for part in tr.name.split(",", 1))
        tr.name, tr.legendgroup = h_label, h_label
        tr.showlegend = family == primary
    for label, dash in families:
        fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", name=label,
                                 legendgroup=label, showlegend=True,
                                 line=dict(color=T["muted"], width=2, dash=dash)))
    return fig


def two_family_legend(fig, solid_label, dashed_label):
    """Colour carries the model size, dash the family: one legend entry per size, plus two keys."""
    return family_legend(fig, [(solid_label, "solid"), (dashed_label, "dash")])


def _stagger(ends, lo, hi, gap_frac=0.05):
    """Push labels apart so they never collide, without pushing any of them off the axis.

    Labels are placed from the top down with a minimum gap of ~5% of the plotted range (less if
    they would not all fit), then pushed back up from `lo` wherever that cascade ran past it.
    """
    span = (hi - lo) or 1.0
    gap = min(gap_frac * span, span / max(len(ends) - 1, 1))
    order = sorted(ends, reverse=True)
    ys = []
    for y_val, _ in order:                      # top down: at least `gap` below the label above
        ys.append(y_val if not ys else min(y_val, ys[-1] - gap))
    for i in reversed(range(len(ys))):          # bottom up: on the axis, `gap` above the one below
        ys[i] = max(ys[i], lo if i == len(ys) - 1 else ys[i + 1] + gap)
    return {key: y for (_, key), y in zip(order, ys)}


def label_line_ends(fig, df, x, y, color, *, logx=False, log_default=True,
                    fmt=lambda key: f"H={key}", colours=None):
    """Direct labels at the right end of each line, laid out for BOTH y scales.

    Plotly reads annotation coordinates as log10 of the value on a log axis, so each label gets a
    log position and a linear one; `y_scale_buttons` swaps between them.
    """
    ends, palette, xs = [], dict(colours or {}), {}
    for i, (key, part) in enumerate(df.groupby(color, sort=False)):
        last = part.sort_values(x).iloc[-1]
        ends.append((last[y], key))
        palette.setdefault(key, T["series"][i % len(T["series"])])
        xs[key] = log10(last[x]) if logx else last[x]

    lin = _stagger(ends, df[y].min(), df[y].max())
    positive = df[y][df[y] > 0]                       # a log axis cannot show a zero series
    floor = positive.min() if len(positive) else 1.0
    log = _stagger([(log10(max(v, floor)), k) for v, k in ends],
                   log10(floor), log10(max(df[y].max(), floor)))
    order = [k for _, k in sorted(ends, reverse=True)]
    for key in order:
        fig.add_annotation(x=xs[key], y=(log if log_default else lin)[key],
                           text=fmt(key), showarrow=False, xanchor="left", xshift=8,
                           font=dict(size=12, color=palette[key]))
    return {"log": [log[k] for k in order], "linear": [lin[k] for k in order]}


def y_scale_buttons(fig, positions, *, active="log", ranges=None):
    """A log / linear switch for the y axis that also moves the direct labels.

    `ranges` gives an explicit y range per scale, for axes (like a 0-100% share) whose fixed
    range would otherwise be read as decades once the axis goes log.
    """
    def args(kind):
        out = {"yaxis.type": kind, "yaxis.dtick": 1 if kind == "log" else None}
        if ranges:
            out["yaxis.range"] = ranges[kind]
        for i, y in enumerate(positions[kind]):
            out[f"annotations[{i}].y"] = y
        return [out]

    fig.update_layout(updatemenus=[dict(
        type="buttons", direction="right", showactive=True, x=0, xanchor="left",
        y=1.06, yanchor="bottom", pad=dict(t=0, b=0),
        bgcolor=T["surface"], bordercolor=T["grid"], borderwidth=1,
        font=dict(size=12, color=T["text"]), active=0 if active == "log" else 1,
        buttons=[dict(label="log y", method="relayout", args=args("log")),
                 dict(label="linear y", method="relayout", args=args("linear"))],
    )])
    fig.update_yaxes(type=active, dtick=1 if active == "log" else None,
                     **({"range": ranges[active]} if ranges else {}))
    return fig


def token_ticks(fig, values):
    """Readable ticks on a log context axis: 256, 1k, 4k, ... instead of plotly's minor ladder."""
    def tick(v):
        if v >= 1 << 20:
            return f"{v >> 20}M"
        return f"{v // 1024}k" if v >= 1024 else str(v)

    fig.update_xaxes(tickmode="array", tickvals=values, ticktext=[tick(v) for v in values],
                     minor_showgrid=False, showticklabels=True)
    return fig

In [ ]:
rows = [inference_flops(VOCAB, H, L, variant=v, **KW).as_row() | {"model": name}
        for name, v in FAMILIES for H in H_VALUES for L in CONTEXTS]
df = pd.DataFrame(rows)
df["TFLOPs"] = df["total"] / 1e12
df["size"] = df["H"].map(SIZES)

fig = px.line(df, x="context_length", y="TFLOPs", color="size", line_dash="model",
              log_x=True, color_discrete_sequence=T["series"],
              line_dash_map={VANILLA: "solid", AME: "dash"},
              category_orders={"size": [SIZES[H] for H in H_VALUES], "model": [VANILLA, AME]},
              hover_data={"size": False, "context_length": ":,", "TFLOPs": ":.3f",
                          "attention_share": ":.1%"})
fig.update_traces(line_width=2, mode="lines")
two_family_legend(fig, VANILLA, AME)
pos = label_line_ends(fig, df[df["model"] == VANILLA], "context_length", "TFLOPs", "size",
                      logx=True, fmt=str)
token_ticks(fig, CONTEXT_TICKS)
style(fig, title="Cost of one forward pass over the full context",
      subtitle=f"vanilla (GQA) vs AME of the same H, {FF_NOTE}, vocab {VOCAB:,} · AME pays "
               f"{AME_PREFILL_COST:.0%} of the dense score path · solid = vanilla, dashed = AME "
               f"· log x",
      xtitle="context length (tokens)", ytitle="TFLOPs per prefill",
      legend_title="vanilla parameters")
y_scale_buttons(fig, pos, active="log")   # decades by default; the buttons switch to linear
fig.update_yaxes(minor_showgrid=False)
fig.show()

In [ ]:
# Table view of the same figure - accessibility fallback and the numbers to quote.
table = (df[df["context_length"].isin(CONTEXT_TICKS)]
           .pivot_table(index="context_length", columns=["model", "size"], values="TFLOPs")
           .rename_axis(index="context length").round(2))
table[[VANILLA, AME]]

### The cost of one generated token

The figure above is a whole prefill, so it grows with the context simply because there are more
tokens in it - and it draws AME and the vanilla model almost on top of each other, because an AME
prefill *is* dense attention: at the same `H` it is 4.5-6.0% cheaper at 4k tokens (the deleted
`W_Q`/`W_K` matmuls) and 0.1-0.3% cheaper at 1M, where the shared score path is nearly the whole bill.

This one is **one decode step with a warm cache**: the FF, the projections and the head run for a
single token. The vanilla model compares that token's `q` against every cached `k`, scales every
cached `v` by those scores and sums them, so its score work is `4 · datt · L` per layer - linear in
the context, not quadratic. AME reads the scores off the token's row of the precomputed table
instead, sums only its top-k values, and scores every cached bigram to find the top-k; `miss` is the
share of steps that compute the row from the banks instead, 0 by default:

```
vanilla:  2 · (61440 H³ + 64 V H)  +  256 H² L                                   FLOPs per token
AME:      2 · (57344 H³ + 64 V H)  +  128 H² · (min(L, k) + miss · U(L))  +  3 H² L
```

(the prefill's `128 H² L²` carries a 1/2 from `pairs = L(L+1)/2`; a decode step attends to all
`L` cached keys, so it has no such factor). A vanilla prefill of `L` tokens is the sum of these steps
for 1..L, which is why it ends up quadratic while each step is linear. An AME prefill is **not** the
sum of its decode steps: it is computed as ordinary attention, and the score table, top-k and
V-cache are decode-only machinery (design §2.2).

With the table, the value sum stops growing once the cache holds `k` tokens and the bigram
selection keeps growing at `3 H²` per cached token, 1/85 of the vanilla model's `256 H²`. That puts an
AME step at 1.45 GFLOPs at `H = 16`, against the vanilla model's 69.3, at 1M tokens. Computing every
row from the banks would add a third term, growing until the context holds every type in the vocab
(past ~450k) and flat after, and raise the step to 3.09.

In [ ]:
dec = pd.DataFrame([inference_flops(VOCAB, H, L, mode="decode", variant=v, **KW).as_row()
                    | {"model": name}
                    for name, v in FAMILIES for H in H_VALUES for L in CONTEXTS])
dec["GFLOPs"] = dec["total"] / 1e9
dec["size"] = dec["H"].map(SIZES)

fig = px.line(dec, x="context_length", y="GFLOPs", color="size", line_dash="model",
              log_x=True, color_discrete_sequence=T["series"],
              line_dash_map={VANILLA: "solid", AME: "dash"},
              category_orders={"size": [SIZES[H] for H in H_VALUES], "model": [VANILLA, AME]},
              hover_data={"size": False, "context_length": ":,", "GFLOPs": ":.2f",
                          "attention_share": ":.1%"})
fig.update_traces(line_width=2, mode="lines")
two_family_legend(fig, VANILLA, AME)
pos = label_line_ends(fig, dec[dec["model"] == VANILLA], "context_length", "GFLOPs", "size",
                      logx=True, fmt=str)
token_ticks(fig, CONTEXT_TICKS)
style(fig, title="Cost of generating one token, with the context already cached",
      subtitle=f"one decode step · {AME_NOTE} · solid = vanilla, dashed = AME · log x",
      xtitle="tokens already in the KV cache", ytitle="GFLOPs per generated token",
      legend_title="vanilla parameters")
y_scale_buttons(fig, pos, active="log")
fig.show()

In [ ]:
# Same data as a table: AME levels off, the vanilla model does not.
(dec[dec["context_length"].isin(CONTEXT_TICKS)]
    .pivot_table(index="context_length", columns=["model", "size"], values="GFLOPs")
    .rename_axis(index="cached tokens").round(2))[[VANILLA, AME]]

#### The same figure as a ratio

Dividing the two lines above gives how many times more a decode step costs on the vanilla model
than on AME of the same size. In units of one cached key's vanilla score work, `256 H²`, it is exactly

```
ratio(L) = (L + 480 H + V/(2H)) / (448 H + V/(2H) + (min(L, k) + miss · U(L)) / 2 + 3 L / 256)
```

`480 H + V/(2H)` is the vanilla weight bill in those units and `448 H + V/(2H)` AME's (the
`61440 / 57344 = 1.07` of the blocks, plus the shared head), so every line leaves the left edge at
**1.07-1.08**. From there the numerator grows with the whole cache and the denominator only with
half of what AME sums and with its bigram scores, so the ratio climbs. Once the top-k is full the
only term still growing below the line is the bigram selection, so the ratio has a ceiling: **256/3,
about 85x**, the vanilla cost of a cached key over AME's.

How close a model gets to it is set by its weights. With the table (`miss = 0`, the default) the
value-sum term is `k / 2 = 1,024` key-units, and at a 1M-token cache the ratio runs from **29.6x**
for the largest default model to **48.0x** for the smallest. Computing every row (`miss = 1`) would
make that term `(k + V) / 2`, about 26k key-units at long contexts - a large share of a small model's
weight bill - and the same spread would be 17.5x to 22.4x. Without the shift the selection term and
the ceiling go, and the spreads would be 44.8-108.3x and 21.9-30.3x. Either way the smaller the model
the bigger the win: the score path AME skips grows as `H²` while the weights it still pays grow as
`H³`.

In [ ]:
rat = (dec.pivot_table(index=["H", "context_length"], columns="model", values="GFLOPs")
          .reset_index())
rat["ratio"] = rat[VANILLA] / rat[AME]
rat["size"] = rat["H"].map(SIZES)

fig = px.line(rat, x="context_length", y="ratio", color="size",
              log_x=True, color_discrete_sequence=T["series"],
              category_orders={"size": [SIZES[H] for H in H_VALUES]},
              hover_data={"size": False, "context_length": ":,", "ratio": ":.2f"})
fig.update_traces(line_width=2, mode="lines")
pos = label_line_ends(fig, rat, "context_length", "ratio", "size", logx=True, fmt=str)
token_ticks(fig, CONTEXT_TICKS)
style(fig, title="How many times a generated token costs more on vanilla than on AME",
      subtitle=f"the two lines above, divided · {AME_NOTE} · log x",
      xtitle="tokens already in the KV cache", ytitle="vanilla GFLOPs / AME GFLOPs",
      legend_title="vanilla parameters")
y_scale_buttons(fig, pos)
fig.show()

### Where attention starts to dominate

The weight terms are linear in tokens and the score path is quadratic, so there is a context length
`L*` at which they cost the same. Setting the two equal (GQA, gated FF, `pairs ~ L²/2`):

```
vanilla:                     128 H² L² = L (122880 H³ + 128 V H)  ->  L* = 960 H + V / H
AME at AME_PREFILL_COST c: c 128 H² L² = L (114688 H³ + 128 V H)  ->  L* = (896 H + V / H) / c
```

`L*` is **not** monotone in `H`. The first term is the blocks (which grow as `H³` against the
score path's `H²`) and the `V / H` term is the LM head (fixed vocab, so it fades as the model
grows). They trade off at `H = sqrt(V / 960)`, about **H = 7** for a 50k vocab: below that the head
dominates and attention bites early, above it every step up in `H` pushes the crossover *later*.

AME pays the full score path in a prefill, so its crossover sits just **below** the vanilla one - at
`896 H` instead of `960 H`, because without `W_Q` and `W_K` its weights are the smaller counterweight.
The figure below is therefore two near-identical families.

Decode is where they part. A vanilla decode step is half attention once `L = 480 H + V/(2H)`, half the
prefill's `L*`, since the newest token attends to all `L` keys rather than `L/2` of them on average.
An AME step is half attention once `min(L, k) + miss · U(L) + 3 L / 128` reaches `896 H + V / H` - the
same number as its prefill crossover, counted in positions summed, row entries computed (none with
the table) and bigrams scored. The bigram term keeps growing, so every size crosses eventually: the
last column below lands at 658k-1.87M tokens for the default sizes, and would land at 80k-297k with
every row computed.

In [ ]:
share = df.copy()
share["attention_share_pct"] = share["attention_share"] * 100

fig = px.line(share, x="context_length", y="attention_share_pct", color="size",
              line_dash="model", log_x=True, color_discrete_sequence=T["series"],
              line_dash_map={VANILLA: "solid", AME: "dash"},
              category_orders={"size": [SIZES[H] for H in H_VALUES], "model": [VANILLA, AME]},
              hover_data={"size": False, "context_length": ":,", "attention_share_pct": ":.1f"})
fig.update_traces(line_width=2, mode="lines")
two_family_legend(fig, VANILLA, AME)
pos = label_line_ends(fig, share[share["model"] == VANILLA], "context_length",
                      "attention_share_pct", "size", logx=True, log_default=False, fmt=str)
token_ticks(fig, CONTEXT_TICKS)
style(fig, title="Share of a prefill spent in the attention score path",
      subtitle="QKᵀ and A@V only · solid = vanilla, dashed = AME",
      xtitle="context length (tokens)", ytitle="% of total FLOPs",
      legend_title="vanilla parameters")
y_scale_buttons(fig, pos, active="linear",       # a share axis reads best linear; log is a click away
                ranges={"linear": [0, 100], "log": [-2, 2]})
fig.update_yaxes(ticksuffix="%")
fig.show()

In [ ]:
def crossover_exact(H, *, variant="vanilla", mode="prefill", vocab=VOCAB, hi=1 << 24):
    """Smallest context at which the score path is at least half the pass; None if it never is
    below `hi`. attention_share is monotone in L for both families and modes, so bisect it."""
    def share(L):
        return inference_flops(vocab, H, L, mode=mode, variant=variant, **KW).attention_share
    if share(hi) < 0.5:
        return None
    lo = 1
    while lo < hi:
        mid = (lo + hi) // 2
        if share(mid) < 0.5:
            lo = mid + 1
        else:
            hi = mid
    return lo


def tokens(L):
    return "never" if L is None else f"{L:,}"


def ame_crossover(H, c=AME_PREFILL_COST):
    return "never" if c == 0 else f"{round((896 * H + VOCAB / H) / c):,}"


pd.DataFrame([{"H": H, "vanilla params": size_label(H),
               "vanilla L* = 960H + V/H": f"{round(960 * H + VOCAB / H):,}",
               "vanilla L* (exact)": tokens(crossover_exact(H)),
               "AME L* = (896H + V/H) / c": ame_crossover(H),
               "AME L* (exact)": tokens(crossover_exact(H, variant="ame")),
               "vanilla decode L* = 480H + V/(2H)": f"{round(480 * H + VOCAB / (2 * H)):,}",
               "AME decode L* (exact)": tokens(crossover_exact(H, variant="ame", mode="decode"))}
              for H in [4, 8] + H_VALUES])  # 4 and 8 included to show the minimum near H = 7

### What the FLOPs are spent on

One model size at four context lengths, each shown for both families - once for a whole prefill and
once for a single decode step.

In the prefill the two bars of each context are nearly the same bar: AME gives up a slice of the
attention projections and nothing else, so at 1M tokens both are 98% attention. In the decode step
they separate. The vanilla step is 99% attention at a 1M-token cache, its `QKᵀ` and `A@V` over a million
keys; the AME step at the same cache costs 1.45 GFLOPs against 69.3, of which 56% is the bigram
selection and 5% the top-k value sum, and the rest weights. Computing every row from the banks would
make it 3.09 GFLOPs, 53% of it the row.

In [ ]:
TERMS = ["feedforward", "attn_projections", "lm_head", "attn_scores", "score_rows", "selection"]
LABELS = {"feedforward": "gated feed-forward (gate, up, down)" if GATED_FF else "feed-forward (up, down)",
          "attn_projections": "attention projections",
          "lm_head": "LM head", "attn_scores": "attention pair work (QKᵀ, A@V)",
          "score_rows": "AME score rows, computed from the banks",
          "selection": "AME top-k selection, bigram scores"}


def breakdown(mode):
    """One size at four contexts, both families, as shares of a prefill or of one decode step."""
    passes = {(L, name): inference_flops(VOCAB, H_PICK, L, mode=mode, variant=v, **KW)
              for L in CONTEXTS_BD for name, v in FAMILIES}
    unit, scale = ("TFLOPs", 1e12) if mode == "prefill" else ("GFLOPs", 1e9)
    terms = [t for t in TERMS if any(getattr(f, t) for f in passes.values())]  # drop all-zero terms
    order = [f"{L:,} · {name}" for L in CONTEXTS_BD for name, _ in FAMILIES]
    bd = pd.DataFrame([
        {"row": f"{L:,} · {name}", "term": LABELS[t],
         "share": getattr(f, t) / f.total * 100, unit: getattr(f, t) / scale}
        for (L, name), f in passes.items() for t in terms
    ])

    fig = px.bar(bd, x="share", y="row", color="term", orientation="h",   # a term keeps its colour
                 color_discrete_map={LABELS[t]: T["series"][i] for i, t in enumerate(TERMS)},
                 category_orders={"term": [LABELS[t] for t in terms], "row": order},
                 hover_data={"row": False, "share": ":.1f", unit: ":.2f"})
    fig.update_traces(marker_line_color=T["surface"], marker_line_width=2)  # 2px surface gap
    for (L, name), f in passes.items():  # absolute cost, since shares alone hide the growth
        fig.add_annotation(x=100, y=f"{L:,} · {name}", text=f"{f.total / scale:,.1f} {unit}",
                           showarrow=False, xanchor="left", xshift=10,
                           font=dict(size=12, color=T["text"]))
    prefill = mode == "prefill"
    style(fig, title=(f"Where {'a prefill pass' if prefill else 'one generated token'} spends its "
                      f"FLOPs (H = {H_PICK})"),
          subtitle=(f"dmodel {64 * H_PICK}, {H_PICK} layers, {FF_NOTE} · vanilla "
                    f"{size_label(H_PICK)} vs AME {size_label(H_PICK, 'ame')}"
                    + ("" if prefill else f" · {AME_NOTE}")),
          xtitle=f"% of the {'pass' if prefill else 'step'}",
          ytitle=f"{'context length' if prefill else 'cached tokens'} · model", legend_title="")
    fig.update_layout(barmode="stack", hovermode="closest", height=520,
                      margin=dict(l=170, r=150, t=100, b=120),
                      legend=dict(orientation="h", x=0, y=-0.22, yanchor="top"))
    fig.update_xaxes(range=[0, 100], ticksuffix="%", showgrid=False)  # a share axis: linear only
    return fig


breakdown("prefill").show()
breakdown("decode").show()

### Without a stored table: computing the rows

Every figure above reads the score row from the precomputed table. The design record does not store
it (§2.1): at `V²` entries per head and layer it would be 5.06 GB in bf16 per head-layer and 50.6 TB
over the 10,000 of `H = 100`, so the design keeps only the banks, computes a row from them when a step
needs it, and caches the rows of frequent query tokens - a cache §4.4 calls "load-bearing, not an
optimization". These sweeps price that alternative: AME per generated token with 0, 14, 25, 50 and
100% of its rows computed - 0% is the table, 14% and 25% the design's 10k- and 3k-type LRU caches -
against a vanilla model, starting at the same size (`H = 50`).

At the same size AME is the cheaper of the two at every context and every share. Below 32k tokens
the share barely registers - 15.6 to 17.8 GFLOPs per token at 32k, against the vanilla model's
36.7. Past that the lines fan out until the context holds every type, which bends them at about
450k, then rise together with the bigram selection: at 1M they sit at 23.2 GFLOPs with the table and
39.3 with every row computed, against 687 - 30x and 17x cheaper.

In [ ]:
REFERENCE = "vanilla"


def row_sweep(h_ame, h_ref):
    """AME per generated token with a share of its score rows computed from the banks instead of
    read from the table, against a vanilla model."""
    rows = ([inference_flops(VOCAB, h_ame, L, mode="decode", variant="ame",
                             **(KW | {"row_miss": m})).as_row() | {"miss": f"{m:.0%}"}
             for m in SWEEP_ROW_MISS for L in CONTEXTS]
            + [inference_flops(VOCAB, h_ref, L, mode="decode", **KW).as_row() | {"miss": REFERENCE}
               for L in CONTEXTS])
    data = pd.DataFrame(rows)
    data["GFLOPs"] = data["total"] / 1e9

    order = [f"{m:.0%}" for m in SWEEP_ROW_MISS] + [REFERENCE]
    colours = {label: T["series"][i] for i, label in enumerate(order[:-1])} | {REFERENCE: T["muted"]}
    ref_label = f"vanilla {size_label(h_ref)}"

    fig = px.line(data, x="context_length", y="GFLOPs", color="miss", log_x=True,
                  color_discrete_map=colours, category_orders={"miss": order},
                  hover_data={"miss": False, "context_length": ":,", "GFLOPs": ":.2f",
                              "attention_share": ":.1%"})
    fig.update_traces(line_width=2, mode="lines")
    for tr in fig.data:  # the reference stays solid, the AME lines are dashed as everywhere else
        tr.line.dash = "solid" if tr.name == REFERENCE else "dash"
        if tr.name == REFERENCE:
            tr.name = ref_label
    pos = label_line_ends(fig, data, "context_length", "GFLOPs", "miss", logx=True,
                          colours=colours,
                          fmt=lambda key: ref_label if key == REFERENCE
                          else "table" if key == f"{0:.0%}" else f"{key} computed")
    token_ticks(fig, CONTEXT_TICKS)
    same = h_ame == h_ref
    style(fig,
          title=(f"What the precomputed table saves (AME at H = {h_ame})" if same
                 else f"AME at H = {h_ame} against vanilla at H = {h_ref}, per generated token"),
          subtitle=(f"AME {size_label(h_ame, 'ame')}, {TOPK_NOTE}, score row computed from the banks "
                    f"on {SWEEP_ROW_MISS[0]:.0%}-{SWEEP_ROW_MISS[-1]:.0%} of steps instead of read "
                    f"from the table · dashed = AME, solid = vanilla · log x"),
          xtitle="tokens already in the KV cache", ytitle="GFLOPs per generated token",
          legend_title="rows computed")
    return y_scale_buttons(fig, pos, active="log")


row_sweep(*SWEEP_PAIRS[0]).show()

### A bigger AME against a smaller vanilla model

AME at `H = 50` has a 7.49B backbone, plus 16.1B of banks, against vanilla `H = 40`'s 4.19B, so it does
1.80x the weight work per token and starts out more expensive: 15.3 against 9.8 GFLOPs per token at a
4k cache. The vanilla model's score path overturns that at about **18k tokens** with the table and
22k with every row computed, and by 1M AME costs 23.2-39.3 GFLOPs per token against 438 - 11-19x less.

In [ ]:
row_sweep(*SWEEP_PAIRS[1]).show()

### The same comparison one size class up

`H = 100` AME (a 57.99B backbone and 64.4B of banks) against `H = 80` vanilla (31.97B), with 1.82x the
weight work per token. The pattern repeats with everything shifted right, because both the weight gap
and the score path grow with `H`: AME is cheaper past about **34k tokens** with the table and 40k
with every row computed, and at 1M it costs 149-214 GFLOPs per token against 1,781 - 8.3-12x less.

In [ ]:
row_sweep(*SWEEP_PAIRS[2]).show()

### And the same again at the 1T scale

`H = 252` AME (a 919.32B backbone and 408.9B of banks) against `H = 200` vanilla (492.81B) - 1.87x the
weight work, the widest gap of the three pairs, and 1,854-1,866 against 1,026 GFLOPs per token at a 4k
cache. The crossover moves out to about **87k tokens** with the table and 99k with every row
computed, and at 1M it is 2,053-2,462 against 11,722 GFLOPs per token, a factor of 4.8-5.7.

That factor shrinks with every size class - 11-19x, 8.3-12x, 4.8-5.7x - even though the contexts and
the row settings are identical. Weight work grows as `H³` and the score path only as `H²`, so the
term AME skips is a smaller slice of a bigger model, and the extra weights AME carries are a larger
one. Scale alone erodes the advantage; only longer contexts restore it.

In [ ]:
row_sweep(*SWEEP_PAIRS[3]).show()

### AMEm: the vanilla model's weights, AME's attention

`AMEm` is AME with the weights its attention layers no longer need put back into the feed-forward.
Dropping `q_proj` and `k_proj` saves `1.5 d²` per layer and the full-width `v_proj` takes `0.5 d²` of
it back, which leaves `d²`: that buys `d / 3` more `dff` with a gated FF (`d / 2` ungated). Rounded to
a multiple of 32 it is `dff = 4448` at `H = 16` and `13856` at `H = 50`, about `4.33 dmodel`, and it
matches the vanilla parameter count outside the banks to within 0.15% at these sizes - exactly where
`d / 3` is itself a multiple of 32 (`H = 24`), and at every `H` with an ungated FF.

The match is on everything *but* the banks because the banks alone are 2-5x the rest of the model at
these sizes, so no `dff` could bring AME's total down to the vanilla one; they cost no FLOPs, though,
so matching the rest matches the weight work per token. AMEm therefore does what the vanilla model
does per token except attention. Its prefill is the vanilla prefill - both are dense attention - so
its figures are per generated token, drawn in the vanilla model's colours: the three families sit
together at a short cache and part as soon as the vanilla score path grows.

In [ ]:
tri = pd.DataFrame([inference_flops(VOCAB, H, L, mode="decode", variant=v, **KW).as_row()
                    | {"model": name}
                    for name, v in FAMILIES3 for H in H_VALUES for L in CONTEXTS])
tri["GFLOPs"] = tri["total"] / 1e9
tri["size"] = tri["H"].map(SIZES)

DASHES = {VANILLA: "solid", AME: "dash", AMEM: "dot"}
fig = px.line(tri, x="context_length", y="GFLOPs", color="size", line_dash="model",
              log_x=True, color_discrete_sequence=T["series"], line_dash_map=DASHES,
              category_orders={"size": [SIZES[H] for H in H_VALUES],
                               "model": [VANILLA, AME, AMEM]},
              hover_data={"size": False, "context_length": ":,", "GFLOPs": ":.3f",
                          "attention_share": ":.1%"})
fig.update_traces(line_width=2, mode="lines")
family_legend(fig, list(DASHES.items()))
pos = label_line_ends(fig, tri[tri["model"] == VANILLA], "context_length", "GFLOPs", "size",
                      logx=True, fmt=str)
token_ticks(fig, CONTEXT_TICKS)
style(fig, title="All three families per generated token",
      subtitle=f"colour = the vanilla parameter count · solid = vanilla, dashed = AME, "
               f"dotted = AMEm · {AME_NOTE} · log x",
      xtitle="tokens already in the KV cache", ytitle="GFLOPs per generated token",
      legend_title="vanilla parameters")
y_scale_buttons(fig, pos, active="log")
fig.update_yaxes(minor_showgrid=False)
fig.show()

#### The same thing as a ratio

Each AME and AMEm line divided by the vanilla line of the same `H` and the same cache. Two readings
the absolute figure cannot give:

* **The dotted curve is AME's attention priced against the vanilla model's**, since AMEm does the
  vanilla weight work. At a 256-token cache it is 98.7-99.5% of the vanilla step - AME's value sum
  and selection over that few keys cost a little less than the vanilla `QKᵀ` and `A@V` - and it
  falls below half past 12k-27k tokens, to 2.1-3.5% at 1M.
* **The gap between the dotted and the dashed line of one colour is AME's weight saving**, constant in
  GFLOPs - 0.035 per token at `H = 16`, 1.01 at `H = 50` - so as a share it is largest at a short cache,
  where AME is 93.0-93.3% of the vanilla step, and fades as the shared attention terms grow.

In [ ]:
ratio = tri.pivot_table(index=["H", "size", "context_length"], columns="model",
                        values="GFLOPs").reset_index()
ratio = ratio.melt(id_vars=["H", "size", "context_length", VANILLA], var_name="model",
                   value_name="GFLOPs")
ratio["% of vanilla"] = ratio["GFLOPs"] / ratio[VANILLA] * 100

fig = px.line(ratio, x="context_length", y="% of vanilla", color="size", line_dash="model",
              log_x=True, color_discrete_sequence=T["series"],
              line_dash_map={AME: "dash", AMEM: "dot"},
              category_orders={"size": [SIZES[H] for H in H_VALUES], "model": [AMEM, AME]},
              hover_data={"size": False, "context_length": ":,", "% of vanilla": ":.2f",
                          "GFLOPs": ":.2f"})
fig.update_traces(line_width=2, mode="lines")
family_legend(fig, [(AMEM, "dot"), (AME, "dash")])
pos = label_line_ends(fig, ratio[ratio["model"] == AMEM], "context_length", "% of vanilla", "size",
                      logx=True, log_default=False, fmt=str)
token_ticks(fig, CONTEXT_TICKS)
if ratio["% of vanilla"].min() < 50 < ratio["% of vanilla"].max():   # only if a curve crosses it
    fig.add_hline(y=50, line_dash="dot", line_color=T["muted"], line_width=1,
                  annotation_text="half the vanilla cost", annotation_position="bottom left",
                  annotation_font=dict(size=11, color=T["muted"]))
style(fig, title="AME and AMEm per generated token, as a share of the vanilla model of the same H",
      subtitle=f"dotted = AMEm, dashed = AME · {AME_NOTE} · log x",
      xtitle="tokens already in the KV cache", ytitle="% of the vanilla decode step",
      legend_title="vanilla parameters")
y_scale_buttons(fig, pos, active="linear")
fig.update_yaxes(ticksuffix="%", minor_showgrid=False)
fig.show()

### Against a vanilla model that does not run all-to-all attention

Every figure so far charges the vanilla model for **every** causal pair, which no deployed
long-context model actually does. [`../notes/long-context-attention.md`](../notes/long-context-attention.md)
works through what the published alternatives cost; each one reduces to a **retention** `c(L)` - the
share of the dense pair work it still does - which drops straight into the model above. A method has
two retentions, because a prefill and a decode step are different passes: for a prefill, the share of
the `L(L+1)/2` causal pairs it scores; for one decode step, the share of the `L` cached keys the newest
token attends to. For most methods the second is the first read at the last query - but **MInference**
sparsifies the prefill only and decodes densely.

The ten solid curves below are the vanilla model at `H = 80` under each method, at its published
hyperparameters. The two references are **AME at `H = 100`** (dashed) and **AMEm at `H = 80`**
(dotted), which does exactly the reference's weight work, so the gap from it to a solid line is the
difference between the two attentions alone. There are two figures: per prefill, then per generated
token.

**What each line is**, ordered by its prefill retention at 1M:

| line | what it does | `c` at 1M, prefill | `c` at 1M, decode |
|---|---|---:|---:|
| dense | every causal pair, `L(L+1)/2` of them. FlashAttention and Ring Attention compute exactly this - they change memory traffic and parallel decomposition, not FLOPs | 100% | 100% |
| Gemma 2 1:1 | alternating layers: one sliding window of 4096, one full-context. The window term dies as `2w/L`, the global layers do not, so it settles at half | 50.39% | 50.20% |
| linear 3:1 | three layers of linear attention (zero pairs, a fixed-size recurrent state) per one full-attention layer | 25.00% | 25.00% |
| MoBA | blocks of 4096, each query routed to its top 12 by a parameter-free gate, 29 of 32 layers sparse. The 3 full layers are 9.4 points of either column; the budget of 49,152 keys means it decodes densely below that | 17.67% | 13.62% |
| Gemma 3 5:1 | five local layers of window 1024 per global layer - the window shrank 4x and the ratio went 1:1 -> 5:1, both reported at minimal perplexity cost | 16.83% | 16.75% |
| linear 7:1 | seven linear layers per softmax layer. Jamba is literally this (1 attention per 8); MiniMax-01 phrases it as one softmax block after every seven | 12.50% | 12.50% |
| NSA | three gated branches: stride-16 compressed blocks (a quadratic term divided by 16, hence the floor), top-16 selected blocks of 64, and a window of 512. Trained in from scratch | 6.54% | 6.40% |
| DSA | a small FP8 "lightning indexer" scores every pair cheaply (5.9% of the dense cost, and that is the floor), then full attention runs over the top 2048 only | 6.29% | 6.10% |
| MInference | one of three patterns per head, fixed offline: A-shape, vertical-slash, or block-sparse - about 4k columns and diagonals per query. A serving-time retrofit of the **prefill**, no retraining; decoding stays dense | 0.78% | 100% |
| local only | a 1024 window in every layer and no global layer: linear in `L`, and the only line with nothing left to pay asymptotically | 0.20% | 0.10% |

The legend names models that ship each one, from their model cards and technical reports as of
2026-10-20. Two caveats belong with those names: MInference is a retrofit onto an already-trained
model rather than a model family, and MiniMax-M2 is listed under *dense* because MiniMax is
reported to have dropped linear attention after M1.

#### Where Mamba and the other SSMs sit

They are not a separate line: a Mamba/SSM layer scores **no pairs at all** - `c = 0` - so a pure SSM's
curve is the reference's weights alone, a line the figures leave out. What it pays instead is a
per-token constant: the recurrence on a fixed state of `d_inner x N` costs about `5 d_inner N ~ 10 d N`
FLOPs per token per layer, which against an H-balanced model's `122880 H³` of weight work per token is
exactly `N / (192 H)`:

| state `N` | share of the weight work at `H = 80` |
|---|---:|
| 16 (Mamba-1) | 0.10% |
| 128 (Mamba-2) | 0.83% |
| 256 | 1.67% |

Under 2% of the pass and flat in the context, so in FLOP terms sequence mixing is nearly free: a
prefill is linear in `L` (the SSD form does matmuls inside chunks of <=128 and a recurrence between
them) and a decode step is **constant** in `L` rather than linear. The saving our charts cannot
show is the memory: the state is `256 N H²` bytes, i.e. **210 MB per sequence at `H = 80`,
`N = 128`, and it does not grow**, against a KV cache of 800 KiB per token - 859 GB at 1M tokens.

What a fixed state costs is retrieval capacity, not FLOPs, which is why nothing ships pure: Jamba
is 1 attention layer per 8 (the `linear 7:1` line exactly), Nemotron-H keeps ~8% of its layers as
attention (4 of 52 at 8B, 10 of 118 at 56B, so just below that line), and Qwen3-Next and Kimi K3
use 3:1 with gated-delta-rule linear attention rather than an SSM.

AME sits elsewhere on both axes. Its decode step is not constant in `L`: the value sum levels off
and the score row is a table read, but the bigram selection keeps growing at `3 H²` per cached
token - 0.019 GFLOPs per 1k tokens at `H = 80`, 1/85 of the vanilla rate - and what levels off is
already larger than an SSM's recurrence: a top-2048 value sum alone is `2 · datt · 2048` per layer,
1.68 GFLOPs per token at `H = 80` against Mamba-2's 0.52. And its memory grows: AME drops the K-cache but
keeps a **full-width** V-cache, `datt` per layer per token - at `H = 80` exactly the 800 KiB per token
of the GQA model's half-width K and V together - and the precomputed table is a fixed `V²` entries per
head and layer on top, 32.4 TB at `H = 80` in bf16. So AME does not share an SSM's open question, a fixed state's
retrieval capacity; its own is that its scores depend on token ids and distance only (design §3).

Quality is **not** modelled in any of this: nine of the ten lines are approximations of attention
(only the dense line is exact), and AME is a different attention, not an approximation of dense.
These figures answer the narrower question of what each one costs.

In [ ]:
def _pairs(L):
    return L * (L + 1) / 2


def _window(L, w):      # prefill pairs under a sliding window of w keys per query
    return _pairs(L) if L <= w else L * w - w * (w - 1) / 2


def _budget(L, b):      # prefill pairs under a fixed budget of b keys per query (top-k selection)
    return _pairs(L) if L <= b else b * (b + 1) / 2 + (L - b) * b


# (short label, models that ship it, what it does, prefill retention, decode retention).
# Retention is the share of the dense pair work a method still does: of the L(L+1)/2 causal pairs
# for a prefill, of the L cached keys for one decode step. The model attributions are from model
# cards and technical reports as of 2026-10-20 and are sourced one by one in
# ../notes/long-context-attention.md, together with the papers and the quality evidence.
ATTENTION_METHODS = [
    ("dense", "Llama 3.1, DeepSeek-V3.1, MiniMax-M2",
     "every causal pair - FlashAttention and Ring Attention compute this exactly",
     lambda L: 1.0, lambda L: 1.0),
    ("Gemma 2 1:1", "Gemma 2",
     "1 local : 1 global layer, window 4096",
     lambda L: (_window(L, 4096) / _pairs(L) + 1) / 2,
     lambda L: (min(L, 4096) / L + 1) / 2),
    ("Gemma 3 5:1", "Gemma 3, Gemma 4",
     "5 local : 1 global layer, window 1024",
     lambda L: (5 * _window(L, 1024) / _pairs(L) + 1) / 6,
     lambda L: (5 * min(L, 1024) / L + 1) / 6),
    ("linear 7:1", "MiniMax-01, MiniMax-M1, Jamba",
     "7 linear-attention layers : 1 softmax layer",
     lambda L: 1 / 8, lambda L: 1 / 8),
    ("linear 3:1", "Kimi Linear, Kimi K3, Qwen3-Next",
     "3 linear-attention layers : 1 full-attention layer",
     lambda L: 1 / 4, lambda L: 1 / 4),
    ("MoBA", "Kimi",
     "blocks of 4096, top-12 per query, 29 of 32 layers sparse",
     lambda L: (29 * _budget(L, 12 * 4096) / _pairs(L) + 3) / 32,
     lambda L: (29 * min(L, 12 * 4096) / L + 3) / 32),
    ("NSA", "DeepSeek 27B research model",
     "stride-16 compression + 16x64 selection + window 512",
     lambda L: min(1.0, 1 / 16 + _budget(L, 16 * 64 + 512) / _pairs(L)),
     lambda L: min(1.0, 1 / 16 + min(L, 16 * 64 + 512) / L)),
    ("DSA", "DeepSeek-V3.2, GLM-5, GLM-5.2, GLM-5.3",
     "top-2048 selection + a lightning indexer at 5.9% of the dense pair cost",
     lambda L: min(1.0, 0.059 + _budget(L, 2048) / _pairs(L)),
     lambda L: min(1.0, 0.059 + min(L, 2048) / L)),
    ("MInference", "retrofit onto Yi-200K, GLM-4-9B-1M",
     "~4k columns and diagonals per query, chosen offline per head - prefill only, dense decode",
     lambda L: _budget(L, 4096) / _pairs(L), lambda L: 1.0),
    ("local only", "Mistral 7B, Character.AI",
     "sliding window 1024, no global layer (those two use wider windows)",
     lambda L: _window(L, 1024) / _pairs(L), lambda L: min(L, 1024) / L),
]


def sparse_pass(H, L, c, mode):
    """A vanilla pass that does only the share `c` of its dense pair work."""
    f = inference_flops(VOCAB, H, L, mode=mode, **KW)
    return f.total - (1 - c) * f.attn_scores


def ame_pass(H, L, mode, variant="ame"):
    return inference_flops(VOCAB, H, L, mode=mode, variant=variant, **KW).total

In [ ]:
H_AME, H_REF = METHOD_PAIR
AME_LINE = f"AME {size_label(H_AME, 'ame')}"
AMEM_LINE = f"AMEm {size_label(H_REF, 'amem')}"   # the reference's weights, AME's attention

# two references in neutral ink: the bigger AME dashed, AMEm at the reference's size dotted
REFS = {AME_LINE: (H_AME, "ame", "dash", T["muted"]), AMEM_LINE: (H_REF, "amem", "dot", T["text"])}

# the legend names the models that ship each method; the line ends keep the short label
LEGEND = {short: f"{short} ({models})" for short, models, *_ in ATTENTION_METHODS}
SHORT = {name: short for short, name in LEGEND.items()} | {k: k for k in REFS}


def methods_figure(mode):
    """The vanilla reference under every method, against AME and AMEm."""
    prefill = mode == "prefill"
    unit, scale = ("TFLOPs", 1e12) if prefill else ("GFLOPs", 1e9)
    pick = 3 if prefill else 4                     # which retention function of each method
    detail = f"prefill at {AME_PREFILL_COST:.0%} of the dense score path" if prefill else AME_NOTE
    rows = [{"method": LEGEND[m[0]], "detail": m[2], "context_length": L, "retention": m[pick](L),
             unit: sparse_pass(H_REF, L, m[pick](L), mode) / scale}
            for m in ATTENTION_METHODS for L in CONTEXTS]
    rows += [{"method": name, "detail": detail, "context_length": L, "retention": float("nan"),
              unit: ame_pass(H, L, mode, variant) / scale}
             for name, (H, variant, _, _) in REFS.items() for L in CONTEXTS]
    data = pd.DataFrame(rows)

    order = [LEGEND[m[0]] for m in ATTENTION_METHODS] + list(REFS)
    colours = ({LEGEND[m[0]]: T["series_long"][i] for i, m in enumerate(ATTENTION_METHODS)}
               | {name: colour for name, (*_, colour) in REFS.items()})
    fig = px.line(data, x="context_length", y=unit, color="method", log_x=True,
                  color_discrete_map=colours, category_orders={"method": order},
                  hover_data={"method": False, "detail": True, "context_length": ":,",
                              unit: ":.1f", "retention": ":.2%"})
    fig.update_traces(line_width=2, mode="lines")
    for tr in fig.data:      # the two references keep their own dash, every method is solid
        tr.line.dash = REFS[tr.name][2] if tr.name in REFS else "solid"
    pos = label_line_ends(fig, data, "context_length", unit, "method", logx=True,
                          colours=colours, fmt=SHORT.get)
    token_ticks(fig, CONTEXT_TICKS)
    style(fig, title=(f"Long-context attention methods {'per prefill' if prefill else 'per generated token'}"
                      f": vanilla H = {H_REF} against AME H = {H_AME}"),
          subtitle=("each solid line is the vanilla model doing only the pair work its method keeps "
                    f"· dashed = AME, dotted = AMEm at the vanilla H = {H_REF} weights · log x"),
          xtitle="context length (tokens)" if prefill else "tokens already in the KV cache",
          ytitle="TFLOPs per prefill" if prefill else "GFLOPs per generated token",
          legend_title="attention (models that ship it)")
    fig.update_layout(margin=dict(l=70, r=360, t=130, b=60))   # room for the labels and the legend
    y_scale_buttons(fig, pos, active="log")
    fig.update_yaxes(minor_showgrid=False)
    return fig


methods_figure("prefill").show()

#### The same methods, per generated token

The prefill figure above has nothing for AME to win: an AME prefill is dense attention, so at
`H = 100` it costs more than the dense `H = 80` line at every context (1.58x at 1M) and more than every
sparse method, and AMEm at `H = 80` sits on the dense line. This is the comparison AME was designed
for - one decode step, with each method's decode retention, and AME's top-k value sum and bigram
selection, its score rows read from the table.

In [ ]:
methods_figure("decode").show()

#### Where AME still wins

AME at `H = 100` does 1.82x the weight work of vanilla `H = 80` per token, and that gap is constant in
the context. A method that attends to `B` keys per query pays `256 H² B` per generated token for its
scores, so it costs more than AME once `B` passes `(AME - vanilla weights) / 256 H²`: at a 1M-token
cache that is **52,479 keys**, against `B* = 480 H + V/(2H) = 38,714`, the budget at which the vanilla
model's score path equals its own weight bill.

The consequence is in the table's crossover columns. Per generated token AME beats dense attention past
34k tokens - and with it MInference, which decodes densely, and MoBA, which is dense up to its 49k
budget - Gemma 2's 1:1 schedule past 65k, the 3:1 linear hybrids past 144k, Gemma 3's 5:1 past 219k
and the 7:1 hybrids past 312k. The trained-in sparse designs are the close call: NSA and DSA score
`L/16` and `0.059 L` keys per query in their compression branch and indexer - 65.5k and 61.9k at 1M,
plus their selections - which puts them above AME's 52.5k at 1M, where they cost 1.16x and 1.13x as
much as AME, but AME overtakes them only past 718k and 768k. A plain 1k window it **never** beats:
65 GFLOPs per token against AME's 149 at 1M. Per prefill it beats none of them, at any context up to
16.8M.

Computing the rows instead of reading them moves exactly the close calls. With every row computed
AME costs 214 GFLOPs per token at 1M, `B` rises to 91,779 keys, and NSA and DSA stay cheaper until
1.61M and 1.73M (at 1M AME costs 1.23x and 1.27x as much); at the design's 3k-type LRU (25% of rows
computed) the same crossovers are 941k and 1.01M.

The dotted line asks the question at equal weights. AMEm at `H = 80` does the reference's weight work,
63.4 GFLOPs per token, so the gap from it to a solid line is the difference between the two attentions
alone: at 1M AME's attention costs 21.8 GFLOPs on top of those weights (1.34x in all) and dense
attention's 1,718 (28.1x). AMEm beats every method at every context but three - the 3:1 hybrids past
4k, the 7:1 hybrids past 9k, and a 1k window never - and at 1M it is 2.03x and 1.97x cheaper per token
than NSA and DSA.

Of AME's own 2.36x over those weights at `H = 100`, 1.82x is its bigger weights, 0.50x the bigram
selection and 0.04x the value sum - which is why, at long contexts, the comparison that matters is
with NSA and DSA, and the costs that matter are the ones no FLOP count shows: storing and reading the
table, and the memory traffic of the bigram scan.

In [ ]:
GRID = sorted({round(2 ** (k / 8)) for k in range(8 * 24 + 1)})   # 1 .. 2^24, 1/8 octave apart


def cheaper_past(cheaper):
    """The context past which `cheaper(L)` holds at every grid point up to 2^24 (~16.8M tokens),
    refined by bisection inside the last bracket; None if it does not hold at 2^24."""
    if not cheaper(GRID[-1]):
        return None
    worse = [L for L in GRID if not cheaper(L)]
    if not worse:
        return 1
    lo, hi = worse[-1] + 1, GRID[GRID.index(worse[-1]) + 1]
    while lo < hi:
        mid = (lo + hi) // 2
        if cheaper(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo


def wins_past(c_fn, mode, h=None, variant="ame"):
    """The context past which AME (or AMEm) at `h` stays cheaper than the vanilla reference
    running the method with retention `c_fn`."""
    h = h or H_AME
    L = cheaper_past(lambda L: ame_pass(h, L, mode, variant)
                     < sparse_pass(H_REF, L, c_fn(L), mode))
    return f"> {GRID[-1] / 1e6:.1f}M" if L is None else f"{L:,}"


LONG = CONTEXT_RANGE[1]
WEIGHTS = sparse_pass(H_REF, LONG, 0.0, "decode")     # the reference's weights alone, per token


def per_token(c_fn):
    return sparse_pass(H_REF, LONG, c_fn(LONG), "decode")


pd.DataFrame([{"method": f"{short} - {detail}",
               "models": models,
               f"c at {LONG:,}, prefill": f"{c_pre(LONG):.2%}",
               f"c at {LONG:,}, decode": f"{c_dec(LONG):.2%}",
               f"vanilla GFLOPs per token at {LONG:,}": round(per_token(c_dec) / 1e9, 1),
               "x its weights": round(per_token(c_dec) / WEIGHTS, 2),
               "x AME, decode": round(per_token(c_dec) / ame_pass(H_AME, LONG, "decode"), 2),
               "x AMEm, decode": round(per_token(c_dec) / ame_pass(H_REF, LONG, "decode", "amem"), 2),
               "AME cheaper past": wins_past(c_dec, "decode"),
               "AMEm cheaper past": wins_past(c_dec, "decode", H_REF, "amem"),
               "x AME, prefill": round(sparse_pass(H_REF, LONG, c_pre(LONG), "prefill")
                                       / ame_pass(H_AME, LONG, "prefill"), 2)}
              for short, models, detail, c_pre, c_dec in ATTENTION_METHODS])

## Using it elsewhere

```python
from h_balanced_flops import inference_flops, total_params   # after `jupyter nbconvert --to script`
```

or copy the functions - they have no dependencies beyond the standard library.

* **AME**: `variant="ame"` on `inference_flops`, `weight_params` and `total_params`. It drops `q_proj`
  and `k_proj`, widens `v_proj` to the full `datt`, and adds `2 V datt` of FLOP-free bank parameters per
  layer (`weight_params(...)["qk_banks"]`; `total_params(..., banks=False)` leaves them out) plus its
  sinks and per-head scalars (`"ame_extras"`). Its score path: `prefill_cost` (1.0 = ordinary
  attention, as designed) and, for a decode step, `top_k` (2048; `None` sums every value), `row_miss`
  (0.0 = every row read from the precomputed table, 1.0 = every row computed from the banks), `shift`
  (True = bigram selection) and `heaps` (the distinct-type estimate, used when rows are computed).
* **AMEm**: `variant="amem"` - the same, with `dff` widened by `d/3` (gated) or `d/2` (ungated), rounded
  to a multiple of 32, so everything but the banks matches the vanilla model of the same `H`.
* **No attention cost at all**, what this notebook first assumed:
  `variant="ame", prefill_cost=0, top_k=0, row_miss=0, shift=False`.
* **One generated token** with a warm cache: `inference_flops(V, H, L, mode="decode")` - the cost of
  producing the token at position `L`, not of the whole context. A vanilla prefill of `L` tokens equals
  the sum of the `L` decode steps (asserted above), so it is ~`L` times one step; an AME prefill does
  not, because it is computed as ordinary attention.
* **Ungated FF** (`up` + `down` only, as the current `def_300`/`def_800` configs are written):
  `gated_ff=False`.
* **Count the softmax**: `softmax_flops_per_score=5` (exp + max-subtract + sum + divide, roughly).

A caveat worth repeating when these numbers meet a wall-clock measurement: FLOPs are not time.
The score path is memory-bound at long contexts and the decode step is bandwidth-bound almost
everywhere, so a model with half the FLOPs does not run twice as fast. This bites AME hardest: its
score row costs no FLOPs only because the table holding it is precomputed - `V²` entries per head and
layer, 50.6 TB in bf16 at `H = 100`, against 129 GB for the banks it is computed from - its bigram
selection is a scan over every cached position, and its top-k value sum is ~786k independent 256-byte
gathers per token at the design's reference config (§4.4), so its real decode cost is set by memory
bandwidth and by where the table lives, which no FLOP count shows.